# Grind on "Doc2LoRA" — naive toy version

Paper: https://arxiv.org/pdf/2602.15902

**Core idea (as I read it):** instead of fine-tuning a model with gradient descent on each new task, train a *hypernetwork* that reads a description (a "doc") of the task and directly **emits LoRA weights** that adapt a frozen base model to that task. The hypernet has "learned to optimize" — one forward pass replaces a full SGD run.

## Minimal setup here

- **Base model**: a tiny MLP `f_θ : R² → R` with frozen weights `θ`.
- **LoRA adapter**: low-rank update `ΔW = B @ A` (rank `r`) applied to the first hidden layer.
- **Task family**: 8 binary arithmetic ops on `(a, b)` — `a+b`, `a-b`, `a*b`, `2a+b`, `a-2b`, `|a-b|`, `max(a,b)`, `min(a,b)`. A *single* frozen MLP cannot solve all of these at once, so per-task adaptation is required.
- **"Doc"**: a short text sentence describing the task, e.g. `"compute a plus b"`. A tiny char-level encoder turns it into an embedding.
- **Hypernet `h_φ`**: doc embedding → flattened `(A, B)`.
- **Meta-training**: sample task → encode doc → emit LoRA → run adapted MLP on a batch of `(a,b, y)` → MSE → backprop only through `φ` (and the encoder). `θ` stays frozen.

If it works, at test time we hand the model a *new* doc and it solves the task with **zero gradient steps** on that task.

In [1]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


'cpu'

In [2]:
# --- Task family -------------------------------------------------------------
# Each task = (text description, python function). We sample (a, b) ~ U[-2, 2].

TASKS = [
    ("compute a plus b",                lambda a, b: a + b),
    ("compute a minus b",               lambda a, b: a - b),
    ("compute a times b",               lambda a, b: a * b),
    ("compute two a plus b",            lambda a, b: 2 * a + b),
    ("compute a minus two b",           lambda a, b: a - 2 * b),
    ("compute absolute difference",     lambda a, b: (a - b).abs()),
    ("compute the maximum of a and b",  lambda a, b: torch.maximum(a, b)),
    ("compute the minimum of a and b",  lambda a, b: torch.minimum(a, b)),
]

def sample_batch(task_idx: int, n: int = 64):
    a = torch.empty(n, 1).uniform_(-2, 2)
    b = torch.empty(n, 1).uniform_(-2, 2)
    x = torch.cat([a, b], dim=-1)
    y = TASKS[task_idx][1](a, b)
    return x.to(device), y.to(device)

# Quick sanity check
x, y = sample_batch(2, 4)
print(x, y)

tensor([[-0.0150, -0.7703],
        [ 1.0729,  0.5363],
        [-1.6461, -0.0396],
        [-1.4719,  1.5858]]) tensor([[ 0.0115],
        [ 0.5754],
        [ 0.0652],
        [-2.3341]])


In [3]:
# --- Base MLP with LoRA on the first layer ----------------------------------
# f(x) = W2 · ReLU( (W1 + B@A) x + b1 ) + b2
# W1, b1, W2, b2 are FROZEN. Only (A, B) are produced by the hypernet.

IN_DIM   = 2
HIDDEN   = 32
OUT_DIM  = 1
LORA_R   = 4

class BaseMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.W1 = nn.Parameter(torch.empty(HIDDEN, IN_DIM))
        self.b1 = nn.Parameter(torch.zeros(HIDDEN))
        self.W2 = nn.Parameter(torch.empty(OUT_DIM, HIDDEN))
        self.b2 = nn.Parameter(torch.zeros(OUT_DIM))
        nn.init.kaiming_uniform_(self.W1, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W2, a=math.sqrt(5))

    def forward(self, x, A=None, B=None):
        # A: (r, in_dim), B: (hidden, r)
        W1 = self.W1
        if A is not None:
            W1 = W1 + B @ A
        h = F.relu(F.linear(x, W1, self.b1))
        return F.linear(h, self.W2, self.b2)

base = BaseMLP().to(device)
for p in base.parameters():
    p.requires_grad = False  # frozen

# Count LoRA params we need to emit
LORA_PARAM_COUNT = LORA_R * IN_DIM + HIDDEN * LORA_R
LORA_PARAM_COUNT

136

In [4]:
# --- Doc encoder + Hypernet --------------------------------------------------
# 1) Char-level: build a vocab over all task descriptions.
# 2) Tiny embed + mean-pool -> doc vector.
# 3) MLP head -> flat LoRA params -> reshape to (A, B).

CHARS = sorted({c for desc, _ in TASKS for c in desc})
STOI  = {c: i + 1 for i, c in enumerate(CHARS)}   # 0 = pad
VOCAB = len(STOI) + 1
MAX_LEN = max(len(d) for d, _ in TASKS)

def encode_doc(text: str) -> torch.Tensor:
    ids = [STOI.get(c, 0) for c in text][:MAX_LEN]
    ids = ids + [0] * (MAX_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

DOC_IDS = torch.stack([encode_doc(d) for d, _ in TASKS]).to(device)  # (T, L)

class Doc2LoRA(nn.Module):
    def __init__(self, d_embed=32, d_hidden=128):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d_embed, padding_idx=0)
        self.trunk = nn.Sequential(
            nn.Linear(d_embed, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_hidden), nn.ReLU(),
        )
        # Two separate heads for A and B. LoRA convention: zero-init B so the
        # initial update B@A is exactly zero, but A has normal init so gradient
        # can still flow through B's head. (If we zeroed BOTH heads' last layer
        # we'd also kill gradients into the trunk -> nothing learns.)
        self.head_A = nn.Linear(d_hidden, LORA_R * IN_DIM)
        self.head_B = nn.Linear(d_hidden, HIDDEN * LORA_R)
        nn.init.zeros_(self.head_B.weight)
        nn.init.zeros_(self.head_B.bias)

    def forward(self, doc_ids):
        mask = (doc_ids != 0).float().unsqueeze(-1)         # (B, L, 1)
        e = self.emb(doc_ids) * mask                        # (B, L, D)
        v = e.sum(1) / mask.sum(1).clamp(min=1.0)           # (B, D)
        h = self.trunk(v)
        A = self.head_A(h).view(-1, LORA_R, IN_DIM)
        B = self.head_B(h).view(-1, HIDDEN, LORA_R)
        return A, B

hyper = Doc2LoRA().to(device)
sum(p.numel() for p in hyper.parameters())

38952

In [5]:
# --- Meta-training loop ------------------------------------------------------
# Each step: pick a task, encode its doc -> (A,B), run adapted MLP, MSE, step.
# Only `hyper` parameters get gradients; `base` is frozen.

opt = torch.optim.Adam(hyper.parameters(), lr=3e-3)
STEPS = 4000
BATCH = 128

losses = []
for step in range(STEPS):
    t = random.randrange(len(TASKS))
    x, y = sample_batch(t, BATCH)
    A, B = hyper(DOC_IDS[t:t+1])           # (1, r, in), (1, hidden, r)
    y_hat = base(x, A=A[0], B=B[0])
    loss = F.mse_loss(y_hat, y)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        recent = sum(losses[-200:]) / 200
        print(f"step {step+1:>5d}  recent MSE = {recent:.4f}")

step   500  recent MSE = 0.3586
step  1000  recent MSE = 0.0373
step  1500  recent MSE = 0.2241
step  2000  recent MSE = 0.0186
step  2500  recent MSE = 0.0961
step  3000  recent MSE = 0.0318
step  3500  recent MSE = 0.0122
step  4000  recent MSE = 0.0148


In [6]:
# --- Evaluation: per-task MSE for (a) frozen base, (b) Doc2LoRA-adapted ------

@torch.no_grad()
def eval_task(t, n=2048):
    x, y = sample_batch(t, n)
    base_mse = F.mse_loss(base(x), y).item()
    A, B = hyper(DOC_IDS[t:t+1])
    lora_mse = F.mse_loss(base(x, A=A[0], B=B[0]), y).item()
    return base_mse, lora_mse

print(f"{'task':<35s} {'frozen base':>12s} {'Doc2LoRA':>12s}")
for t, (desc, _) in enumerate(TASKS):
    b_mse, l_mse = eval_task(t)
    print(f"{desc:<35s} {b_mse:>12.4f} {l_mse:>12.4f}")

task                                 frozen base     Doc2LoRA
compute a plus b                          3.2800       0.0012
compute a minus b                         2.3992       0.0013
compute a times b                         1.7787       0.0795
compute two a plus b                      7.2088       0.0028
compute a minus two b                     5.8741       0.0005
compute absolute difference               2.8057       0.0007
compute the maximum of a and b            1.6412       0.0019
compute the minimum of a and b            1.6807       0.0015


In [7]:
# --- Generalization probe: paraphrased / unseen docs -------------------------
# The hypernet was trained on the exact 8 strings above. Does it generalize to
# paraphrases? This is a tiny stress test of "doc -> weights" being semantic
# rather than a lookup table. (With a char encoder + 8 training docs, expect
# this to be brittle — that's the honest result.)

PROBES = [
    ("add a and b",                      0),
    ("subtract b from a",                1),
    ("multiply a by b",                  2),
    ("take the larger of a and b",       6),
    ("take the smaller of a and b",      7),
]

@torch.no_grad()
def eval_doc(text, t_ref, n=2048):
    ids = encode_doc(text).unsqueeze(0).to(device)
    A, B = hyper(ids)
    x, y = sample_batch(t_ref, n)
    return F.mse_loss(base(x, A=A[0], B=B[0]), y).item()

for text, t in PROBES:
    mse = eval_doc(text, t)
    print(f"  '{text}'  ->  target task '{TASKS[t][0]}'   MSE={mse:.4f}")

  'add a and b'  ->  target task 'compute a plus b'   MSE=1.0624
  'subtract b from a'  ->  target task 'compute a minus b'   MSE=2.9628
  'multiply a by b'  ->  target task 'compute a times b'   MSE=4.4087
  'take the larger of a and b'  ->  target task 'compute the maximum of a and b'   MSE=0.7593
  'take the smaller of a and b'  ->  target task 'compute the minimum of a and b'   MSE=3.8214


## What this shows / next steps

**Shows.** A single frozen MLP can't do all 8 ops at once, but a hypernet that reads the task description emits a rank-4 LoRA patch that adapts it per-task in **one forward pass** — no test-time gradient descent. That's the Doc2LoRA pitch in miniature.

**Caveats / honest limits of this toy.**
- Only 8 training docs → the char encoder essentially learns a "doc → id → LoRA" lookup. The paraphrase probe usually fails; that's expected.
- LoRA is on a 2→32 layer, so rank 4 is already nearly full-rank. To make rank matter, widen the base MLP.

**Next steps toward the real paper.**
1. Scale the *task distribution*: sample random affine ops `y = w₁ a + w₂ b + c` with `(w₁, w₂, c)` written into the doc as text. Now the doc carries real numeric info the hypernet must parse.
2. Replace the char encoder with a small frozen sentence-transformer; train only the head + LoRA.
3. Move from MLP→Transformer base, apply LoRA to attention `W_q, W_v` like the paper.
4. Compare against the obvious baseline: per-task SGD fine-tuning of the same LoRA — Doc2LoRA should match it with zero test-time gradient steps.